<a href="https://colab.research.google.com/github/Anshika-019/listing-writer/blob/main/listing_writer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q "gradio<6" ollama


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [9]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull gemma3:4b

In [11]:
"""
Listing Writer: turns product details (and an optional photo) into a
ready-to-paste Etsy/eBay listing, using Gemma running locally through Ollama.

Setup:
  1. Install Ollama from https://ollama.com
  2. ollama pull gemma3:4b
  3. pip install gradio ollama
  4. python listing_writer.py
"""

import gradio as gr
import ollama

MODEL = "gemma3:4b"  # swap for a bigger/newer Gemma if your machine can handle it

SYSTEM_PROMPT = """You write product listings for small online sellers on Etsy and eBay.

STRICT RULES:
- Only use facts the seller gave you or that are clearly visible in the photo.
- Never add features, accessories, specs or "what's in the box" items that weren't given.
- Never write as if you personally used the product. No "I love", "I've been impressed".
- Start directly with "TITLE:". No intro line, no closing line.

Reply in exactly this format:

TITLE:
<one title under 140 characters, front-loaded with search keywords>

DESCRIPTION:
<2 short paragraphs, then a bullet list of only the given details>

TAGS:
<13 comma-separated search tags, each under 20 characters>
"""


def write_listing(name, details, platform, tone, photo):
    if not name.strip():
        return "Add a product name to get started."

    prompt = (
        f"Platform: {platform}\n"
        f"Tone: {tone}\n"
        f"Product: {name}\n"
        f"Seller's notes: {details or 'none'}\n"
    )
    if photo:
        prompt += "A product photo is attached. Use what you can see in it.\n"

    message = {"role": "user", "content": prompt}
    if photo:
        message["images"] = [photo]

    try:
        response = ollama.chat(
            model=MODEL,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, message],
        )
        return response["message"]["content"]
    except Exception as e:
        return (
            f"Couldn't reach the model ({e}).\n\n"
            f"Check that Ollama is running and that you've run: ollama pull {MODEL}"
        )


with gr.Blocks(title="Listing Writer", theme=gr.themes.Soft()) as app:
    gr.Markdown(
        "# Listing Writer\n"
        "Describe your product and get a title, description and tags you can paste "
        "straight into your shop. Powered by Gemma, a free open-weight model, "
        "so there's no paid API behind it."
    )
    with gr.Row():
        with gr.Column():
            name = gr.Textbox(label="Product name", placeholder="Hand-thrown ceramic mug")
            details = gr.Textbox(
                label="Details",
                lines=5,
                placeholder="Speckled glaze, holds 12 oz, dishwasher safe, made in Ohio",
            )
            photo = gr.Image(label="Photo (optional)", type="filepath")
            with gr.Row():
                platform = gr.Radio(["Etsy", "eBay"], value="Etsy", label="Platform")
                tone = gr.Radio(
                    ["Friendly", "Professional", "Playful"], value="Friendly", label="Tone"
                )
            button = gr.Button("Write listing", variant="primary")
        with gr.Column():
            output = gr.Textbox(label="Your listing", lines=22, show_copy_button=True)

    button.click(write_listing, [name, details, platform, tone, photo], output)

if __name__ == "__main__":
    app.launch()

/tmp/ipykernel_1633/742214037.py:68: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Listing Writer", theme=gr.themes.Soft()) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b5ecfbdbac7854c282.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s localhost:11434
!ollama list

Ollama is runningNAME         ID              SIZE      MODIFIED       
gemma3:4b    68ceab7e468a    3.4 GB    11 minutes ago    
